# adcsim Kaggle — constrained min + scorecard (DAR1-7)

Full PyRosetta protocol (not a simplified MMFF substitute).

**Upload as a Kaggle Dataset (Internet ON):**
- `ADC_DAR1.pdb` ... `ADC_DAR6.pdb`
- `ADC_DAR7_omit_B229.pdb`
- `ADC_DAR7_omit_B232.pdb`
- `LPP.params` (pre-generated locally; do NOT run molfile_to_params)

Attach that dataset to this notebook. Outputs go to `/kaggle/working/polished_kaggle/`.


In [ ]:
# Install PyRosetta (CPU). Requires Internet ON and license acceptance.
# GPU is unused: MinMover is CPU-only.
%pip -q install pyrosetta-installer rdkit numpy scipy
from pyrosetta_installer import install_pyrosetta
install_pyrosetta()
print("install done")


In [ ]:
# Locate the 8 representative PDBs and LPP.params
# Strip ADD_RING / NU / virtual ring atoms. Coupled maleimide NU angles
# do not match Rosetta ring-conformer tables and crash pose_from_pdb.
from pathlib import Path

DAR_FILES = [
    "ADC_DAR1.pdb",
    "ADC_DAR2.pdb",
    "ADC_DAR3.pdb",
    "ADC_DAR4.pdb",
    "ADC_DAR5.pdb",
    "ADC_DAR6.pdb",
    "ADC_DAR7_omit_B229.pdb",
    "ADC_DAR7_omit_B232.pdb",
]

def find_file(name: str) -> Path:
    roots = [Path("/kaggle/input"), Path("/kaggle/working"), Path(".")]
    hits = []
    for root in roots:
        if not root.exists():
            continue
        hits.extend(root.rglob(name))
    if not hits:
        raise FileNotFoundError(
            f"{name} not found. Upload it to a Kaggle Dataset and attach the dataset."
        )
    return hits[0]


def strip_ring_records(src: Path, dst: Path) -> Path:
    kept = []
    for ln in src.read_text().splitlines():
        s = ln.strip()
        if s.startswith("ADD_RING") or s.startswith("NU "):
            continue
        if s.startswith("CUT_BOND"):
            kept.append("BOND" + ln[len("CUT_BOND"):])
            continue
        if s.startswith("ATOM VC"):
            continue
        if s.startswith("ICOOR_INTERNAL") and " VC" in ln:
            continue
        toks = s.split()
        if toks and toks[0] == "BOND" and any(t.startswith("VC") for t in toks[1:]):
            continue
        kept.append(ln)
    dst.write_text("\n".join(kept) + "\n")
    return dst

PDB_PATHS = {n: find_file(n) for n in DAR_FILES}
PARAMS_RAW = find_file("LPP.params")
OUT = Path("/kaggle/working/polished_kaggle")
OUT.mkdir(parents=True, exist_ok=True)
PARAMS = strip_ring_records(PARAMS_RAW, OUT / "LPP.norings.params")

print("LPP.params (raw):", PARAMS_RAW)
print("LPP.params (no rings):", PARAMS)
for n, q in PDB_PATHS.items():
    print(f"  {n:28s} {q}  {q.stat().st_size/1e6:.1f} MB")


In [ ]:
# Init PyRosetta ONCE with the stripped ligand params.
# HarmonicFunc lives in core.scoring.func (not constraints) in this build.
import pyrosetta
from pyrosetta import pose_from_pdb
from pyrosetta.rosetta.core.id import AtomID
from pyrosetta.rosetta.core.kinematics import MoveMap
from pyrosetta.rosetta.core.scoring import ScoreFunctionFactory, ScoreType
from pyrosetta.rosetta.core.scoring.constraints import AtomPairConstraint
from pyrosetta.rosetta.core.scoring.func import HarmonicFunc
from pyrosetta.rosetta.protocols.minimization_packing import MinMover

pyrosetta.init(f"-extra_res_fa {PARAMS} -mute all -ignore_unrecognized_res false")
print("pyrosetta init OK", PARAMS)


In [ ]:
# Constrained minimization = polish (not a filter).
# Fixed: protein backbone.
# Mobile: all LPP chi + protein sidechains whose nbr atom is within 14 A of any LPP.
# Constraint: Cys-SG to ligand C91 harmonic 1.82 A, sd=0.05 A.
# Skip a file if its *_min.pdb already exists (resume-safe).

RADIUS_NBR = 14.0  # nbr-atom cutoff; equivalent to ~8 A heavy-atom shell
MAX_ITER = 120


def constrained_min(pdb_path: Path, out_pdb: Path) -> dict:
    pose = pose_from_pdb(str(pdb_path))
    sfxn = ScoreFunctionFactory.create_score_function("ref2015")
    sfxn.set_weight(ScoreType.atom_pair_constraint, 10.0)

    lpp = [i for i in range(1, pose.total_residue() + 1) if pose.residue(i).name3() == "LPP"]
    if not lpp:
        raise RuntimeError(f"no LPP residues in {pdb_path.name}")

    mm = MoveMap()
    mm.set_bb(False)
    mm.set_chi(False)
    mm.set_jump(False)
    for i in lpp:
        mm.set_chi(i, True)

    lpp_xyz = [pose.residue(j).nbr_atom_xyz() for j in lpp]
    n_chi = 0
    for i in range(1, pose.total_residue() + 1):
        if i in lpp:
            continue
        ri = pose.residue(i)
        if not ri.is_protein():
            continue
        xyz = ri.nbr_atom_xyz()
        if any((xyz - q).norm() <= RADIUS_NBR for q in lpp_xyz):
            mm.set_chi(i, True)
            n_chi += 1

    n_cons = 0
    for j in lpp:
        rj = pose.residue(j)
        if not rj.has("C91"):
            continue
        c91 = AtomID(rj.atom_index("C91"), j)
        best, best_d = None, 9e9
        for i in range(1, pose.total_residue() + 1):
            ri = pose.residue(i)
            if ri.name3() != "CYS" or not ri.has("SG"):
                continue
            d = (ri.xyz("SG") - rj.xyz("C91")).norm()
            if d < best_d:
                best_d, best = d, AtomID(ri.atom_index("SG"), i)
        if best is not None and best_d <= 2.5:
            pose.add_constraint(AtomPairConstraint(best, c91, HarmonicFunc(1.82, 0.05)))
            n_cons += 1

    mover = MinMover(mm, sfxn, "lbfgs_armijo_nonmonotone", 0.01, True)
    mover.max_iter(MAX_ITER)
    before = sfxn(pose)
    mover.apply(pose)
    after = sfxn(pose)
    pose.dump_pdb(str(out_pdb))
    return {
        "file": pdb_path.name,
        "n_lpp": len(lpp),
        "n_sidechain": n_chi,
        "n_cons": n_cons,
        "score_before": round(float(before), 3),
        "score_after": round(float(after), 3),
        "output": out_pdb.name,
    }


min_rows = []
for name, src in PDB_PATHS.items():
    out = OUT / f"{Path(name).stem}_min.pdb"
    if out.exists() and out.stat().st_size > 1000:
        print(f"[skip] {name} already minimized")
        continue
    print(f"[min] {name} ...", flush=True)
    rec = constrained_min(src, out)
    min_rows.append(rec)
    print(
        f"      LPP={rec['n_lpp']} cons={rec['n_cons']} "
        f"score {rec['score_before']} -> {rec['score_after']}",
        flush=True,
    )

print("minimization records:", len(min_rows))


In [ ]:
# Cheap scorecard on minimized PDBs.
# Terms (do NOT call these Delta-G):
#   rg / span / reach  compactness of all ligand heavy atoms
#   surface_contact    ligand heavy atoms 2.4-4.5 A from protein
#   mmff_stress        vacuum MMFF single-point on concatenated LPP (optional)
#   cdr_min_dist       min distance ligand heavy -> trastuzumab CDR CA
#   local_ca_rmsd      CA RMSD of protein residues within 8 A of ligand vs input

import json
import numpy as np
from scipy.spatial import cKDTree

# Trastuzumab Kabat CDRs. Chains A/B = heavy, C/D = light in this PDB.
CDR = {
    "A": list(range(31, 36)) + list(range(50, 66)) + list(range(95, 103)),
    "B": list(range(31, 36)) + list(range(50, 66)) + list(range(95, 103)),
    "C": list(range(24, 35)) + list(range(50, 57)) + list(range(89, 98)),
    "D": list(range(24, 35)) + list(range(50, 57)) + list(range(89, 98)),
}


def parse_pdb(path: Path):
    prot, lig, ca, cdr_ca = [], [], {}, []
    for ln in path.read_text().splitlines():
        if not (ln.startswith("ATOM") or ln.startswith("HETATM")):
            continue
        name = ln[12:16].strip()
        resn = ln[17:20].strip()
        chain = ln[21]
        try:
            resi = int(ln[22:26])
        except ValueError:
            continue
        elem = ln[76:78].strip() if len(ln) >= 78 else name[:1]
        if elem == "H":
            continue
        xyz = np.array([float(ln[30:38]), float(ln[38:46]), float(ln[46:54])])
        if ln.startswith("HETATM") or resn == "LPP":
            lig.append(xyz)
        else:
            prot.append(xyz)
            if name == "CA":
                ca[(chain, resi)] = xyz
                if resi in CDR.get(chain, []):
                    cdr_ca.append(xyz)
    return (
        np.asarray(prot) if prot else np.zeros((0, 3)),
        np.asarray(lig) if lig else np.zeros((0, 3)),
        ca,
        np.asarray(cdr_ca) if cdr_ca else np.zeros((0, 3)),
    )


def compactness(xyz: np.ndarray):
    com = xyz.mean(axis=0)
    rg = float(np.sqrt(((xyz - com) ** 2).sum(axis=1).mean()))
    span = float(np.max(np.linalg.norm(xyz[:, None] - xyz[None, :], axis=2)))
    return rg, span


def mmff_stress(path: Path):
    try:
        from rdkit import Chem
        from rdkit.Chem import AllChem
        mol = Chem.MolFromPDBFile(str(path), removeHs=False, sanitize=False)
        if mol is None:
            return None
        mol.UpdatePropertyCache(strict=False)
        m = Chem.AddHs(mol, addCoords=True)
        if not AllChem.MMFFHasAllMoleculeParams(m):
            return None
        ff = AllChem.MMFFGetMoleculeForceField(m)
        return round(float(ff.CalcEnergy()), 1)
    except Exception:
        return None


cards = []
for name in DAR_FILES:
    mini = OUT / f"{Path(name).stem}_min.pdb"
    raw = PDB_PATHS[name]
    if not mini.exists():
        print("missing", mini.name)
        continue
    prot, lig, ca_min, cdr = parse_pdb(mini)
    _, _, ca_raw, _ = parse_pdb(raw)
    rg, span = compactness(lig)
    tree = cKDTree(prot)
    d, _ = tree.query(lig, k=1)
    reach = float(np.max(d))
    contact = int(np.sum((d >= 2.4) & (d <= 4.5)))
    cdr_d = float(cKDTree(cdr).query(lig, k=1)[0].min()) if len(cdr) and len(lig) else None
    keys = [k for k in ca_min if k in ca_raw]
    if keys:
        a = np.stack([ca_min[k] for k in keys])
        b = np.stack([ca_raw[k] for k in keys])
        # local: CA within 8 A of any ligand atom in the minimized pose
        lig_tree = cKDTree(lig)
        local = [i for i, xyz in enumerate(a) if lig_tree.query(xyz)[0] <= 8.0]
        if local:
            da = a[local] - b[local]
            local_rmsd = float(np.sqrt((da ** 2).sum(axis=1).mean()))
        else:
            local_rmsd = 0.0
    else:
        local_rmsd = None
    rec = {
        "file": mini.name,
        "n_lig_atoms": int(len(lig)),
        "rg": round(rg, 2),
        "span": round(span, 2),
        "reach": round(reach, 2),
        "surface_contact": contact,
        "cdr_min_dist": None if cdr_d is None else round(cdr_d, 2),
        "local_ca_rmsd": None if local_rmsd is None else round(local_rmsd, 3),
        "mmff_stress_kcal": mmff_stress(mini),
    }
    cards.append(rec)
    print(rec)

summary = {"score_cards": cards, "note": "scorecard != binding free energy; computational DAR != experimental DAR"}
(OUT / "score_summary.json").write_text(json.dumps(summary, indent=2))
print("wrote", OUT / "score_summary.json")


In [ ]:
# Pack outputs for download from the Kaggle viewer.
import shutil
zip_path = shutil.make_archive("/kaggle/working/adcsim_kaggle_out", "zip", OUT)
print("zip:", zip_path)
print("files:")
for p in sorted(OUT.iterdir()):
    print(f"  {p.name:40s} {p.stat().st_size/1e6:.2f} MB")


## Notes

- This notebook is polish + scorecard only. DAR capacity was already decided on the laptop.
- `LPP.params` was generated locally from residue A1001 of `ADC_DAR6.pdb` with `rdkit-to-params`. Atom names include C91.
- Do not interpret MMFF / ref2015 / surface_contact as experimental Delta-G.
- If a pose fails with unrecognized residue LPP, the attached `LPP.params` is missing or not passed to `-extra_res_fa`.
